In [21]:
# 1. Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [22]:
# 2. Load dataset
df = pd.read_csv("/content/heart_disease_uci.csv")

In [23]:
# 3. Convert target to binary (0 = No disease, 1 = Disease)
df['num'] = df['num'].apply(lambda x: 0 if x == 0 else 1)

In [24]:
# 4. Drop unnecessary columns
df = df.drop(columns=['id', 'dataset'])

In [25]:
# 5. Separate features and target
x = df.drop(columns='num')
y = df['num']

In [26]:
# 6. Handle missing values
# Separate numerical and categorical columns
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
cat_cols = x.select_dtypes(include=['object']).columns

In [27]:
# Fill numerical with mean
x[num_cols] = x[num_cols].fillna(x[num_cols].mean())

In [28]:
# Fill categorical with mode
for col in cat_cols:
    x[col] = x[col].fillna(x[col].mode()[0])

/tmp/ipykernel_8236/788654889.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x[col] = x[col].fillna(x[col].mode()[0])


In [29]:
X = pd.get_dummies(x, drop_first=True)

In [30]:
scaler = StandardScaler()

In [31]:
scaler.fit(X)

StandardScaler()

In [32]:
standardized_data = scaler.transform(X)

In [33]:
print(standardized_data)

[[ 1.00738556  0.69804099  0.31102075 ... -0.5320941  -1.69279243
  -0.51355259]
 [ 1.43203377  1.51176059  0.79771293 ... -0.5320941   0.59073988
  -0.51355259]
 [ 1.43203377 -0.65815834  0.27428927 ... -0.5320941  -1.69279243
   1.94722024]
 ...
 [ 0.15808914 -0.54966239  0.21919204 ... -0.5320941  -1.69279243
  -0.51355259]
 [ 0.4765753   0.          1.70681718 ... -0.5320941   0.59073988
  -0.51355259]
 [ 0.90122351 -0.65815834  0.50386105 ... -0.5320941   0.59073988
  -0.51355259]]


In [34]:
X = standardized_data
Y = df['num']

In [35]:
print(X)

[[ 1.00738556  0.69804099  0.31102075 ... -0.5320941  -1.69279243
  -0.51355259]
 [ 1.43203377  1.51176059  0.79771293 ... -0.5320941   0.59073988
  -0.51355259]
 [ 1.43203377 -0.65815834  0.27428927 ... -0.5320941  -1.69279243
   1.94722024]
 ...
 [ 0.15808914 -0.54966239  0.21919204 ... -0.5320941  -1.69279243
  -0.51355259]
 [ 0.4765753   0.          1.70681718 ... -0.5320941   0.59073988
  -0.51355259]
 [ 0.90122351 -0.65815834  0.50386105 ... -0.5320941   0.59073988
  -0.51355259]]


In [36]:
print(Y)

0      0
1      1
2      1
3      0
4      0
      ..
915    1
916    0
917    1
918    0
919    1
Name: num, Length: 920, dtype: int64


In [37]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state = 42)

In [38]:
print(X.shape, x_train.shape, x_test.shape)

(920, 18) (736, 18) (184, 18)


In [39]:
print(Y.shape, y_train.shape, y_test.shape)

(920,) (736,) (184,)


In [40]:
model = LogisticRegression()

In [41]:
model.fit(x_train, y_train)

LogisticRegression()

In [42]:
prediction = model.predict(x_test)

In [43]:
accuracy = accuracy_score(y_test, prediction)
print("Accuracy : ", accuracy)

Accuracy :  0.8043478260869565


In [44]:
from sklearn.model_selection import GridSearchCV

In [45]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'max_iter': [100, 200, 300]
}

In [46]:
grid = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')

In [47]:
grid.fit(x_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'max_iter': [100, 200, 300], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear']},
             scoring='accuracy')

In [48]:
best_model = grid.best_estimator_

In [49]:
prediction = best_model.predict(x_test)
tuned_accuracy = accuracy_score(y_test, prediction)

In [50]:
print("Best Parameters:", grid.best_params_)
print("Tuned Accuracy:", tuned_accuracy)

Best Parameters: {'C': 0.1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'liblinear'}
Tuned Accuracy: 0.8043478260869565


In [51]:
import pickle

# assuming your trained model is named 'model'
with open('heaart_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)

In [52]:
with open('heart_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)